# Use LocaQ Loss with YOLO

This notebook shows how to replace the classification loss in Ultralytics YOLO with LocaQ Loss. The model still uses its existing bounding-box regression and distribution focal losses.

Clone or download this repository and start Jupyter from the repository root so Python can import the adjacent `locaq_loss.py`. Select a kernel with PyTorch and Ultralytics installed. The thyroid experiment's saved log reports Python 3.10.20, PyTorch 2.5.1+cu121, and Ultralytics 8.4.51. Choose the PyTorch build appropriate for your machine.

Use an Ultralytics detection dataset YAML with training and validation splits. A test split is needed only if you run the optional test-set evaluation.

Choose YOLOv8m or YOLO11m using MODEL_CFG. Both models were evaluated in the manuscript. Other YOLO releases require a compatible training-loss interface.


In [ ]:
from pathlib import Path
import types

import torch
import ultralytics
from ultralytics import YOLO
from locaq_loss import LocaQLoss

MODEL_CFG = "yolov8m.yaml"  # Set to "yolo11m.yaml" for YOLO11m.
DATA_YAML = Path("path/to/data.yaml")
DEVICE = 0 if torch.cuda.is_available() else "cpu"
EPOCHS = 1

print(f"PyTorch {torch.__version__}, Ultralytics {ultralytics.__version__}")
if not DATA_YAML.is_file():
    raise FileNotFoundError(f"Set DATA_YAML to your detection dataset YAML: {DATA_YAML}")


## Attach LocaQ Loss and train

The training-start callback installs the custom criterion on the training model. The one-epoch configuration demonstrates the integration. For the paper's thyroid experiment, see the separately provided training and evaluation notebook.


In [ ]:
def inject_locaq_loss(trainer):
    trainer.model.init_criterion = types.MethodType(
        lambda self: LocaQLoss(
            self,
            beta=1.0,
            vfl_alpha=0.75,
            vfl_gamma=2.0,
            anchor_area=1024.0,
            fn_clamp_max=3.0,
        ),
        trainer.model,
    )
    print("LocaQ Loss configured for the training model.")

model = YOLO(MODEL_CFG)
model.add_callback("on_train_start", inject_locaq_loss)
results = model.train(
    data=str(DATA_YAML.resolve()),
    epochs=EPOCHS,
    imgsz=640,
    batch=8,
    device=DEVICE,
    workers=0,
    seed=42,
    deterministic=True,
    project="runs/locaq",
    name=f"{Path(MODEL_CFG).stem}_example",
)


## Optional test-set evaluation

Run this cell after training if your dataset YAML defines a test split. The loss is used during training. The saved detector is evaluated through the usual prediction and metric pipeline.


In [ ]:
best_checkpoint = Path(model.trainer.best)
if not best_checkpoint.is_file():
    raise FileNotFoundError(f"Training checkpoint not found: {best_checkpoint}")
metrics = YOLO(str(best_checkpoint)).val(
    data=str(DATA_YAML.resolve()), split="test", device=DEVICE, imgsz=640
)
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50–95: {metrics.box.map:.4f}")
